Quick and dirty notebook for simulation of coherent scattering data of magnetic samples in fraunhofer far-field regime

# Import

In [ ]:
# Import general libraries
import numpy as np

# plotting
import matplotlib.pyplot as plt

%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

In [ ]:
# Imports from our own codebase
from scattering_calculator.sample_generator import pattern_generator  # magnetic pattern generators
from scattering_calculator.interactive.interactive_widgets import cimshow  # interactive image viewer
from scattering_calculator.simulation_pipelines import simulation_configuration  # config dataclasses
from scattering_calculator.simulation_pipelines import HologramPipeline, HologramPipelineConfig, HologramPipelineRanges
from fomocid import DATA_ROOT  # project root: one level above the fomocid repo

## Simulation pipeline overview

This notebook simulates coherent X-ray scattering (Fourier transform holography, FTH) from a magnetic thin-film sample. The pipeline runs in the following order:

1. **Experimental geometry** — define the X-ray source energy/polarisation and detector layout (pixel size, distance, beamstop).
2. **Sample** — parse the multilayer material recipe, generate a magnetic domain pattern, and punch the FTH holography mask (object hole + reference holes) into the aperture. Apertures can be layer-aware, conical near the top stack, and rough at the boundary.
3. **Hologram computation** — propagate the beam through the sample using the Jones matrix formalism for both circular-right (CR) and circular-left (CL) polarisations. The optional multislice mode also propagates the Jones field through free space between material layers before projecting the exit wavefield onto the detector; padding and edge-absorber controls reduce FFT wraparound artifacts.
4. **Post-processing** — compute helicity difference/sum, FTH reconstructions, and frame averages.
5. **Export** — save arrays and experimental metadata to an HDF5 file. Optional outputs can include a detected hologram generated from the same detector-noise realization but without the beamstop shadow or detector threshold cap.

### EXPERIMENTAL GEOMETRY

In [ ]:
# ===================
# X-ray Source
# ===================
#######################################
# Create config class for x-ray source
xrayconfig = simulation_configuration.XRayConfig(
    energy           = 787.9, # eV
    pol              = "CR",  # initial polarization (overridden per-helicity in the propagation loop)
    photon_flux      = 1e10, # Photons per second
    coherence_length = (20e-6, 20e-6),  # in m, (y, x); controls partial coherence blur on the detector
)
xrayconfig.setup()  # derives beam_params (wavelength, wavenumber, etc.) from energy

In [ ]:
# ==================
# DETECTOR-BEAMSTOP GEOMETRY
# ==================
detector_pixel_size = 20e-6  # in m
detector_pixel_shape = (1300, 1300)
detector_distance = 0.02  # in m
detector_center = (650, 650)  # in px

# Detector response only. Use readout_noise_sigma, not the legacy noise_rms alias.
detector_params = {
    "readout_noise_average": 50,
    "readout_noise_sigma": 3,
    "detector_threshold": 64e3,
    # Canonical counts-to-photon conversion used by detection and photon artifacts.
    "counts_per_photon": 180,
    "quantum_efficiency": 0.85,
}
# Acquisition timing/frame settings are kept separate from detector response.
measurement_config = {
    "number_frames": 60,
    "max_counts_per_image": 63.5e3, #if None there is no renormalization
    "exposure_time": 5., # s
}
# Photon-event shape controls only; counts_per_photon belongs in detector_params.
artifacts_config = {
    "sigma_photon": 0.75,
    "photon_n_classes": 1,
    "photon_n_variants": 30,
    "photon_kernel_size": 9,
    "photon_irregularity": 2.0,
    "regenerate_photon_kernels": True,
}

# Optional: Define a beamstop
# Basic parameters for beamstop
beamstop_distance = 0.001  # in m
beamstop_center = np.array(detector_pixel_shape) // 2  # in px; centred on the direct beam

# Select beamstop method and parameters
beamstop_method = "circular"  # "circular", "rectangular", None
beamstop_radius = 0.2e-3  # in m
beamstop_sigma = 0.01e-3  # in m
beamstop_wire_width = 0.05e-3  # in m
beamstop_wire_bend = 0.1e-3  # in m
beamstop_theta=np.pi/6
beamstop_ellipticity_range = (0.9,1.1)
beamstop_roughness= 0.05
beamstop_roughness_modes = (3, 9)
beamstop_antialias = 4  # supersample beamstop/wire rasterisation to reduce jagged edges
save_detected_hologram_without_beamstop = False  # optionally save CR/CL detected_no_beamstop

####################################
# Create config class for beamstop
beamstop_config = simulation_configuration.BeamstopConfig(
    bs_method=beamstop_method,
    bs_detector_distance=beamstop_distance,
    bs_center=beamstop_center,
    bs_config={"radius"         : beamstop_radius,
               "angle"          : beamstop_theta,
               "sigma"          : beamstop_sigma,
               "ellipticity"    : beamstop_ellipticity_range,
               "roughness"      : beamstop_roughness,
               "roughness_modes" : beamstop_roughness_modes,
               "wire_width"     : beamstop_wire_width,
               "wire_bend"      : beamstop_wire_bend,
               "antialias"      : beamstop_antialias,
               "seed"           : None,
               },
)

# Create config class for detector
detectorconfig = simulation_configuration.DetectorConfig(
    pixel_size=detector_pixel_size,
    shape=detector_pixel_shape,
    sample_to_detector_distance=detector_distance,
    detector_center=detector_center,
    detector_params=detector_params,
    measurement_config=measurement_config,
    artifacts_config=artifacts_config,
    beamstop_config=beamstop_config
    
)
detectorconfig.setup()
detectorconfig.visualize_beamstop()  # sanity check: confirm beamstop placement before simulation

## This is the resolution we will have thanks to the detector
real_space_pixel_size = (
    detectorconfig.calc_realspace_resolution(xrayconfig.beam_params) / 2  # /2 for 2× oversampling
)



`real_space_pixel_size` is derived from the detector geometry and photon wavelength via the Nyquist criterion: it sets the physical size of one pixel in the sample plane and determines the field of view of the simulation. Dividing by 2 oversamples by a factor of 2, which avoids aliasing in the far-field propagation.

# SAMPLE 

### SAMPLE STACK STRUCTURE AND OPTICAL PROPERTIES

In [ ]:
# ===================
# MATERIAL RECIPE
# ===================
# Slash-separated terms are separate propagated layers. Adjacent terms
# without a slash are combined into one thickness-weighted effective layer,
# e.g. Pt(4)/Co(6) gives two layers, while Pt(4)Co(6) gives one 10 nm layer.
recipe = "[Au(70)/Cr(30)]x8/SiN(100)/[Co(4)Pt(0.2)Al(0.2)]x5"  # layer stack in nm, top to bottom
oversampling = 2  # lateral oversampling factor: sample grid is 2× the detector grid to avoid wrap-around artefacts

sample_shape = np.array([
        0,  # Nz: sentinel — updated automatically by sampleconfig.setup() to match number of layers
        oversampling * detectorconfig.shape[0],
        oversampling * detectorconfig.shape[1],
    ],
    dtype=int,
)  # in pixels

sampleconfig = simulation_configuration.SampleConfig(
    recipe=recipe,
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    xray_config=xrayconfig,
    sample_name="Test_Sample",
)
sampleconfig.setup()

`sample_shape[0] = 0` is a sentinel — it is updated automatically to match the number of layers after `setup()`. The lateral dimensions are set to twice the detector size so that the sample field of view covers the full detector without wrap-around artefacts during the Fourier propagation.

### Magnetic domain pattern

Use this cell for generated magnetic patterns or for an experimental binary image reconstruction. The optional skyrmion cell below is kept as a direct low-level example; run only one magnetic-pattern cell before continuing with the magnetization mapping.

In [ ]:
# --- Magnetic domain pattern ---
# Choose "wavy_stripe_pattern", "binary_labyrinth_pattern", "disordered_skyrmion_lattice_pattern", "saturated_pattern", or "image_pattern".
pattern_type = "binary_labyrinth_pattern"

stripe_width = 20e-9       # stripe width in m, or average skyrmion diameter for skyrmion lattices
sigma = 1e-9               # edge/domain-wall smoothing in m; disordered skyrmions use this as a profile transition width

# Wavy stripe controls
angle_stripes = np.pi / 4  # orientation of stripe wavevector (radians from x-axis)
wave_amplitudes = 80e-9    # peak-to-peak lateral waviness amplitude in m
wave_scale = 20e-9         # spatial period of the waviness modulation in m

# Experimental binary image controls
experimental_pattern_path = None        # e.g. DATA_ROOT / "Data" / "reconstruction_domains.png"
experimental_pattern_pixel_size = None  # real-space pixel size of the reconstruction, in m
experimental_pattern_threshold = 0.5
experimental_pattern_invert = False
experimental_pattern_pad_mode = "edge"

# Saturated-state controls
saturated_config = {
    "saturation": 1,
}

# Disordered skyrmion lattice controls
skyrmion_lattice_config = {
    "skyrmion_density": 0.25,
    "diameter_spread": 0.10 * stripe_width,
    "ellipticity": (0.75, 1.25),
    "roughness": 0.05,
    "roughness_modes": (3, 9),
}

# Labyrinth controls. The continuous field is rescaled to stripe_width,
# optionally smoothed by sigma, then converted to magnetic contrast. With
# auto_size=True, large target stripes use a smaller generated source field
# whenever the later rescale makes that sufficient.
labyrinth_config = {
    "batch": 1,
    "H": 100,
    "W": 100,
    "n_steps": 50,
    "region": "custom",
    "use_gpu": False,
    "k0": 1.0,
    "eps": 0.0,
    "noise_amp": 0.0,
    "domain_conversion": "soft",  # "soft" avoids hard-threshold artifacts; "hard" gives exact +/-1
    "softness": 1.0,
    "auto_size": True,
    "crop_margin": None,
}

pattern_config = {
    "stripe_width": stripe_width,
    "sigma": sigma,
}
if pattern_type == "wavy_stripe_pattern":
    pattern_config.update(
        {
            "waviness_amplitude": wave_amplitudes,
            "waviness_scale": wave_scale,
            "angle_stripes": angle_stripes,
        }
    )
elif pattern_type == "binary_labyrinth_pattern":
    pattern_config.update(labyrinth_config)
elif pattern_type == "disordered_skyrmion_lattice_pattern":
    pattern_config.update(skyrmion_lattice_config)
elif pattern_type == "saturated_pattern":
    pattern_config.update(saturated_config)
elif pattern_type == "image_pattern":
    if experimental_pattern_path is None or experimental_pattern_pixel_size is None:
        raise ValueError(
            "image_pattern requires experimental_pattern_path and "
            "experimental_pattern_pixel_size."
        )
    pattern_config.update(
        {
            "image_path": str(experimental_pattern_path),
            "image_pixel_size": experimental_pattern_pixel_size,
            "threshold": experimental_pattern_threshold,
            "invert": experimental_pattern_invert,
            "pad_mode": experimental_pattern_pad_mode,
        }
    )
else:
    raise ValueError(f"Unknown pattern_type: {pattern_type}")

magnetic_pattern_config = simulation_configuration.MagneticPatternConfig(
    pattern_type_method=pattern_type,
    shape=sample_shape[1:],                  # 2-D lateral shape (Ny, Nx)
    real_space_pixel_size=real_space_pixel_size,
    pattern_config=pattern_config,           # lengths are given in metres
)
magnetic_pattern_config.create_pattern()
magnetic_pattern_config.plot_pattern()


%%time
# --- Option B: skyrmion lattice ---
skyrmion_radius = 3e-9          # radius of a single skyrmion core in m
screening_radius = (
    1.5 * skyrmion_radius        # exclusion radius around each skyrmion (must be even or odd multiple)
)
skyrmion_smoothing = 1e-9        # smoothing applied to the skyrmion profile, in m

number_of_skyrmions = 60000      # target number of skyrmions to place
max_nr_iteration = 100000        # max placement attempts (brute-force packing — reduce if too slow)

magnetic_pattern_config = simulation_configuration.MagneticPatternConfig(
    pattern_type_method="skyrmion_pattern",
    shape=sample_shape[1:],
    real_space_pixel_size=real_space_pixel_size,
    # Physical-length pattern parameters now belong directly in pattern_config.
    pattern_config={
        "skyr_radius": skyrmion_radius,
        "screening_radius": screening_radius,
        "number_skyr": number_of_skyrmions,
        "number_iter": max_nr_iteration,
        "sigma": skyrmion_smoothing,
    },
)
magnetic_pattern_config.create_pattern()
magnetic_pattern_config.plot_pattern()


In [ ]:
magnetic_pattern = magnetic_pattern_config.magnetic_pattern  # 2-D scalar pattern, values in [-1, 1]

# Build a 3-D magnetization vector field (mx, my, mz) from the scalar out-of-plane pattern.
# mx = 0, my = sqrt(1 - mz²)  (in-plane component to preserve |m| = 1), mz = pattern
magnetization = pattern_generator.map_magnetization_to_3d(
    np.zeros_like(magnetic_pattern),                    # mx: no in-plane x-component
    np.sqrt(1 - np.abs(magnetic_pattern) ** 2),         # my: in-plane y-component (unit vector constraint)
    magnetic_pattern,                                    # mz: out-of-plane component
    nr_repeats=sample_shape[0],                          # repeat across all Nz layers
)

sampleconfig.assign_magnetic_pattern(magnetization)



The 2-D scalar pattern is promoted to a 3-D magnetization vector field `(Nz, Ny, Nx, 3)` with in-plane components set to zero and the out-of-plane component equal to the pattern. `Nz` is replicated across all layers — the same magnetic texture is assumed uniform through the magnetic layer stack.

# - holography mask

In [ ]:
# FTH aperture: one large object hole (OH) + two small reference holes (RH)
apertures_radius  = [120e-9, 5e-9, 15e-9]                          # bottom/base hole radii in m
apertures_types   = ["OH", "RH", "RH"]                            # OH = object hole, RH = reference hole
apertures_centers = [(0, 0), (0.35e-6, -0.25e-6), (0.25e-6, 0.18e-6)]  # (y, x) centres in m
apertures_sigma   = [1e-9, 0.1e-9, 0.1e-9]                       # edge-smoothing sigma per hole in m
apertures_angle = [0.0, 0.0, 0.0]                                # radians; OH remains circular
apertures_ellipticity = [1.0, 1.0, 1.0]                          # y/x ratio; keep OH circular
apertures_top_radius_factor = [2.0, 3.0, 3.0]                    # top/base radius ratio
aperture_roughness_amplitude = 5e-9                             # m, target boundary fluctuation
aperture_roughness_period = 5e-9                                # m, target boundary period


def aperture_roughness_from_length(
    radius,
    amplitude=20e-9,
    period=10e-9,
    max_relative_amplitude=0.25,
):
    """Convert physical roughness amplitude/period to relative Fourier settings."""
    relative_amplitude = min(
        float(max_relative_amplitude),
        float(amplitude) / float(radius),
    )
    center_mode = max(1, int(round(2.0 * np.pi * float(radius) / float(period))))
    return relative_amplitude, (max(1, center_mode - 2), center_mode + 2)


apertures_roughness, apertures_roughness_modes = zip(
    *[
        aperture_roughness_from_length(
            radius,
            amplitude=aperture_roughness_amplitude,
            period=aperture_roughness_period,
        )
        for radius in apertures_radius
    ]
)
apertures_roughness = list(apertures_roughness)
apertures_roughness_modes = list(apertures_roughness_modes)
apertures_seed = [-1, -1, -1]

membrane_index = sampleconfig.sample_structure.layer_names.index("SiN")
thickness_OH = np.sum(sampleconfig.sample_structure.layer_thicknesses[:membrane_index])
# Taper only in the upper part of the top stack. The two layers closest to SiN stay cylindrical.
aperture_taper_depth = np.sum(
    sampleconfig.sample_structure.layer_thicknesses[: max(0, membrane_index - 2)]
)

front_aperture_config = simulation_configuration.FrontApertureConfig(
    aperture_method="FTH_circular",
    aperture_shape=sample_shape,
    real_space_pixel_size=sampleconfig.sample_structure.real_space_pixel_size,
    aperture_thicknesses=sampleconfig.sample_structure.layer_thicknesses,  # per-layer thicknesses for RH depth sum
    aperture_config=dict(
        apertures_type=apertures_types,
        apertures_radius=apertures_radius,
        apertures_center=apertures_centers,
        apertures_sigma=apertures_sigma,
        apertures_angle=apertures_angle,
        apertures_ellipticity=apertures_ellipticity,
        apertures_roughness=apertures_roughness,
        apertures_roughness_modes=apertures_roughness_modes,
        apertures_seed=apertures_seed,
        apertures_top_radius_factor=apertures_top_radius_factor,
        aperture_taper_depth=aperture_taper_depth,
        thickness_OH=thickness_OH,  # total depth of OH from surface down to the SiN membrane
    ),
)
front_aperture_config.setup()
front_aperture_config.visualize_aperture()

aperture_mask = front_aperture_config.return_aperture()
sampleconfig.assign_aperture_mask(aperture_mask)



### Aperture detail viewer

Inspect each aperture ROI, including the depth-averaged mask and vertical sections through the aperture centre.

In [ ]:
# Detailed aperture inspection: overview ROI boxes plus vertical sections through each aperture centre.
from matplotlib.patches import Rectangle


def _aperture_roi_slices(aperture_config, aperture_index):
    """Return the y/x ROI slices used for one configured aperture.

    Parameters
    ----------
    aperture_config : FrontApertureConfig
        Configured aperture object with an initialized Apertures3D instance.
    aperture_index : int
        Index of the aperture to inspect.

    Returns
    -------
    roi_slices : tuple[slice, slice]
        Y and X slices around the aperture, matching the ROI helper used to
        build aperture masks.
    """
    cfg = aperture_config.aperture_config
    pixel_size = aperture_config.real_space_pixel_size
    center_m = cfg["apertures_center"][aperture_index]
    radius_m = cfg["apertures_radius"][aperture_index]
    sigma_m = cfg["apertures_sigma"][aperture_index]
    angle = cfg.get("apertures_angle", [0.0] * len(cfg["apertures_radius"]))[aperture_index]
    ellipticity = cfg.get("apertures_ellipticity", [1.0] * len(cfg["apertures_radius"]))[aperture_index]
    roughness = cfg.get("apertures_roughness", [0.0] * len(cfg["apertures_radius"]))[aperture_index]
    top_radius_factor = cfg.get("apertures_top_radius_factor", [2.0] * len(cfg["apertures_radius"]))[aperture_index]
    center_px = (
        np.array(center_m, dtype=float) / pixel_size
        + np.array(aperture_config.aperture_shape[-2:], dtype=float) / 2
    )
    radius_px = float(radius_m) * max(1.0, float(top_radius_factor)) / pixel_size
    sigma_px = None if sigma_m is None else float(sigma_m) / pixel_size
    return aperture_config.aperture._aperture_bbox(
        aperture_config.aperture.shape,
        center_px,
        radius_px,
        sigma=sigma_px,
        angle=angle,
        ellipticity=ellipticity,
        roughness=roughness,
    )


def visualize_aperture_details(aperture_config, aperture_mask):
    """Display aperture projection crops and vertical centre sections.

    Parameters
    ----------
    aperture_config : FrontApertureConfig
        Configured aperture object containing aperture geometry and pixel size.
    aperture_mask : np.ndarray
        Three-dimensional aperture mask with shape ``(Nz, Ny, Nx)``.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Figure containing the overview and per-aperture detail panels.
    """
    cfg = aperture_config.aperture_config
    pixel_size_nm = aperture_config.real_space_pixel_size * 1e9
    projection = np.mean(aperture_mask, axis=0)
    layer_thickness_nm = np.asarray(aperture_config.aperture.layer_thicknesses) * 1e9
    total_depth_nm = float(np.sum(layer_thickness_nm))
    n_apertures = len(cfg["apertures_radius"])
    fig, axes = plt.subplots(
        n_apertures + 1,
        3,
        figsize=(11, min(3.0 * (n_apertures + 1), 11.5)),
        constrained_layout=True,
        squeeze=False,
    )

    ny, nx = projection.shape
    full_extent_nm = [
        (-nx / 2) * pixel_size_nm,
        (nx / 2) * pixel_size_nm,
        (ny / 2) * pixel_size_nm,
        (-ny / 2) * pixel_size_nm,
    ]
    overview = axes[0, 0]
    overview.imshow(projection, cmap="gray", vmin=0, vmax=1, extent=full_extent_nm, interpolation="nearest")
    overview.set_title("aperture projection + ROIs")
    overview.set_xlabel("x (nm)")
    overview.set_ylabel("y (nm)")
    for idx, aperture_type in enumerate(cfg["apertures_type"]):
        y_slice, x_slice = _aperture_roi_slices(aperture_config, idx)
        x0 = (x_slice.start - nx / 2) * pixel_size_nm
        y0 = (y_slice.start - ny / 2) * pixel_size_nm
        width = (x_slice.stop - x_slice.start) * pixel_size_nm
        height = (y_slice.stop - y_slice.start) * pixel_size_nm
        overview.add_patch(Rectangle((x0, y0), width, height, fill=False, linewidth=1.5))
        center_y, center_x = cfg["apertures_center"][idx]
        overview.plot(center_x * 1e9, center_y * 1e9, "+", markersize=8)
        overview.text(center_x * 1e9, center_y * 1e9, f" {idx}: {aperture_type}", va="center")
    axes[0, 1].axis("off")
    axes[0, 2].axis("off")

    for row, aperture_type in enumerate(cfg["apertures_type"], start=1):
        y_slice, x_slice = _aperture_roi_slices(aperture_config, row - 1)
        center_y_m, center_x_m = cfg["apertures_center"][row - 1]
        center_y_px = int(np.clip(round(ny / 2 + center_y_m / aperture_config.real_space_pixel_size), 0, ny - 1))
        center_x_px = int(np.clip(round(nx / 2 + center_x_m / aperture_config.real_space_pixel_size), 0, nx - 1))
        roi_projection = projection[y_slice, x_slice]
        yz_section = aperture_mask[:, y_slice, center_x_px]
        xz_section = aperture_mask[:, center_y_px, x_slice]
        x_extent_nm = [
            (x_slice.start - nx / 2) * pixel_size_nm,
            (x_slice.stop - nx / 2) * pixel_size_nm,
        ]
        y_extent_nm = [
            (y_slice.stop - ny / 2) * pixel_size_nm,
            (y_slice.start - ny / 2) * pixel_size_nm,
        ]

        axes[row, 0].imshow(
            roi_projection,
            cmap="gray",
            vmin=0,
            vmax=1,
            extent=[x_extent_nm[0], x_extent_nm[1], y_extent_nm[0], y_extent_nm[1]],
            interpolation="nearest",
        )
        axes[row, 0].plot(center_x_m * 1e9, center_y_m * 1e9, "+", markersize=8)
        axes[row, 0].set_title(f"{row - 1}: {aperture_type} ROI projection")
        axes[row, 0].set_xlabel("x (nm)")
        axes[row, 0].set_ylabel("y (nm)")

        axes[row, 1].imshow(
            yz_section,
            cmap="gray",
            vmin=0,
            vmax=1,
            aspect="auto",
            extent=[y_extent_nm[0], y_extent_nm[1], total_depth_nm, 0],
            interpolation="nearest",
        )
        axes[row, 1].set_title("vertical section y-z")
        axes[row, 1].set_xlabel("y (nm)")
        axes[row, 1].set_ylabel("depth (nm)")

        axes[row, 2].imshow(
            xz_section,
            cmap="gray",
            vmin=0,
            vmax=1,
            aspect="auto",
            extent=[x_extent_nm[0], x_extent_nm[1], total_depth_nm, 0],
            interpolation="nearest",
        )
        axes[row, 2].set_title("vertical section x-z")
        axes[row, 2].set_xlabel("x (nm)")
        axes[row, 2].set_ylabel("depth (nm)")

    return fig


visualize_aperture_details(front_aperture_config, aperture_mask)


In [ ]:
# Combine refractive indices and magnetization into the dielectric response used by Jones propagation.
# Must be called after assign_magnetic_pattern() and assign_aperture_mask().
# compact=True stores constant per-layer diagonals plus aperture ROI patches instead of
# materializing the full (Nz, Ny, Nx, 2, 2) tensor stack.
sampleconfig.sample_structure.calculate_final_dielectric_tensor(
    use_aperture_roi=True,
    compact=True,
)

The dielectric tensor encodes both the charge and magnetic contributions to the optical response of each layer. It is computed from the refractive indices (loaded for the given X-ray energy) and the magnetization vector field. This is the main input to the Jones propagator.

### HOLOGRAM COMPUTATION

In [ ]:
# Params for gaussian beam
illumination_function = "gaussian"
illumination_center = (0, 0)
illumination_focus_distance = 1e-3  # in m
illumination_fwhm = 0.5e-6  # in m, it is not showing correct fwhm?

illuminationconfig = simulation_configuration.IlluminationConfig(
    XRayConfig=xrayconfig,
    shape=sample_shape[-2:],
    real_space_pixel_size=real_space_pixel_size,
    illumination_function=illumination_function,
    illumination_config={
        "center": illumination_center,
        "distance": illumination_focus_distance,
        "fwhm": illumination_fwhm,
    },
)
illuminationconfig.setup()
illuminationconfig.visualize_illumination()

A Gaussian beam is used to mimic the focused synchrotron spot. `fwhm` controls the beam diameter at the focus; `distance` shifts the beam waist relative to the sample plane (0 = focus on sample). The beam is stored as a complex Jones wavefield.

In [ ]:
# Container that accumulates exit waves and holograms for all polarisations
hologram_config = simulation_configuration.HologramConfig(
    sample_x=sampleconfig.sample_structure.x,
    sample_y=sampleconfig.sample_structure.y,
    detector_layout=detectorconfig.detector_layout,
)

# False: fast historical Jones transmission mode.
# True: multislice mode, with angular-spectrum free-space propagation between material layers.
# Padding reduces periodic FFT wraparound; the absorber is clamped to the padded margin.
# Use edge/reflect padding to avoid introducing a hard zero wall at the crop boundary.
propagate = True
# False: full-field free-space propagation between layers.
# True: ROI multislice approximation. The field first receives the
# zero-spatial-frequency plane-wave phase everywhere. Each padded aperture
# ROI is then propagated locally, and only its correction relative to that
# plane-wave baseline is added back into the full field. Faster than
# full-field multislice, but still approximate because
# true free-space propagation couples all pixels nonlocally.
multislice_propagation_roi = True
# Extra pixels around aperture ROI boxes for the approximate ROI-only free-space step.
# This enlarges the propagated area and gives the ROI correction room to taper
# smoothly back to the plane-wave baseline; propagation_padding_px only pads FFT boundaries.
multislice_propagation_roi_padding_px = 64
# True: keep the default ROI-overlap behavior.
# False: allow only disjoint padded crops from physically separate aperture
# supports to remain separate. Aperture funnels that intersect in the material
# mask and padded propagation crops that overlap are always merged, because
# otherwise shared pixels receive multiple local diffraction corrections.
multislice_propagation_roi_merge_overlaps = True
propagation_padding_px = 128
propagation_padding_mode = "edge"
propagation_absorber_width_px = 64
propagation_absorber_strength = 6.0
propagation_absorber_profile = "cosine"

for i, polarization in enumerate(["CR", "CL"]):
    illuminationconfig.update_polarization(polarization)  # switch beam Jones vector to CR or CL

    samplepropagationconfig = simulation_configuration.SamplePropagatorConfig(
        SampleConfig=sampleconfig,
        IlluminationConfig=illuminationconfig,
        propagator_method="Jones",  # full Jones matrix propagation through the dielectric tensor
        propagator_config={
            "propagate": propagate,
            "propagation_padding_px": propagation_padding_px,
            "propagation_padding_mode": propagation_padding_mode,
            "propagation_absorber_width_px": propagation_absorber_width_px,
            "propagation_absorber_strength": propagation_absorber_strength,
            "propagation_absorber_profile": propagation_absorber_profile,
            "multislice_propagation_roi": multislice_propagation_roi,
            "multislice_propagation_roi_padding_px": multislice_propagation_roi_padding_px,
            "multislice_propagation_roi_merge_overlaps": multislice_propagation_roi_merge_overlaps,
        },
    )
    samplepropagationconfig.setup()  # runs the propagation; result stored internally

    detectorconfig.assign_propagated_wavefront(samplepropagationconfig)  # project exit wave onto detector
    detectorconfig.detect_hologram()  # add Poisson shot noise for the "detected" version
    detectorconfig.hologram_exp.gnomonic_projection()  # correct for curved Ewald sphere geometry

    hologram_config.add_exit_waves(
        {polarization: samplepropagationconfig.return_scalar_wavefield()}
    )
    hologram_config.add_holograms(
        {polarization: detectorconfig.return_ideal_hologram()}, source="ideal"  # noise-free
    )
    hologram_config.add_holograms(
        {
            polarization: detectorconfig.return_detected_hologram(
                store_no_beamstop=save_detected_hologram_without_beamstop
            )
        },
        source="detected",  # beamstop mask + detector threshold + same detector noise draw
    )
    if save_detected_hologram_without_beamstop:
        hologram_config.add_holograms(
            {polarization: detectorconfig.return_detected_hologram_without_beamstop()},
            source="detected_no_beamstop",
        )

# Canonical metadata is assembled from these configured objects in the export section.


The loop runs the full propagation pipeline for each polarisation (CR and CL):
- **Jones propagation** through the sample dielectric tensor → exit wavefield
- **ROI multislice mode** — optional approximation: the full field starts from the plane-wave phase advance, then each padded aperture ROI contributes `local_propagated - plane_wave_baseline` back into the field
- **Overlapping ROI option** — aperture funnels that intersect in the material mask are always merged into one physical ROI. Padded propagation boxes that overlap are also merged even when `multislice_propagation_roi_merge_overlaps=False`, because shared pixels must not receive multiple local FFT corrections
- **ROI padding** — `multislice_propagation_roi_padding_px` expands those boxes and gives the local correction room to taper smoothly back to the plane-wave baseline, which suppresses hard rectangular crop-edge bands. It is especially useful around small reference holes but can make the ROI approximation closer to full-field multislice on small grids
- **Gnomonic projection** from the exit plane onto the curved detector geometry
- **Ideal hologram** — intensity on the detector without detector noise
- **Detected hologram** — one detector-noise realization with the beamstop mask and detector threshold cap applied
- **Optional no-beamstop detected hologram** — saved as `detected_no_beamstop`, using the same photon/readout-noise realization as `detected` but skipping the beamstop mask and detector threshold cap

Results are accumulated in `hologram_config` keyed by helicity.

In [ ]:
# Quick sanity check: display frame-averaged exit wave amplitudes/phases and holograms for CR and CL
hologram_config.visualize_averages()

In [ ]:
hologram_config.compute_differences()    # CR - CL → magnetic contrast
hologram_config.compute_sums()           # CR + CL → charge background
hologram_config.compute_reconstructions() # FTH: fftshift(fft2(fftshift(holo))) for each key
hologram_config.compute_averages()       # average over frames if stacked

In [ ]:
cimshow(hologram_config.detected_holograms['CR']-hologram_config.detected_holograms['CL'], interpolation="nearest")

Post-processing steps:
- **difference** — CR − CL, isolating the magnetic contrast signal
- **sum** — CR + CL, giving the charge (non-magnetic) background
- **reconstructions** — FTH reconstruction via `fftshift(fft2(fftshift(holo)))` for each helicity
- **averages** — mean over frames if a multi-shot stack was simulated

In [ ]:
# Visualise the FTH reconstruction of the helicity difference from the ideal (noise-free) hologram
hologram_config.visualize_reconstruction(source="ideal", helicity="diff")

In [ ]:
# Visualise the FTH reconstruction of the helicity difference from the ideal (noise-free) hologram
hologram_config.visualize_reconstruction(source="detected", helicity="diff")

# Export data

The interactive result is saved with the same canonical HDF5 structure as `HologramPipeline.run()`: file-global settings under `_pipeline_config/`, then arrays and effective metadata under `00000/`. This makes files created here directly compatible with the pipeline usage tutorial and sweep readers.

In [ ]:
# Destination for this single interactively simulated sample.
output_path = DATA_ROOT / "Data" / "simulation_name" / "simulation.h5"
output_path

In [ ]:
# Build the same diagnostics saved by HologramPipeline.run().
supportmask = front_aperture_config.create_supportmask(
    output_shape=detectorconfig.detector_layout.detector_shape,
    output_pixel_size=detectorconfig.detector_layout.real_space_resolution,
)
oh_mask = front_aperture_config.create_supportmask(
    output_shape=sample_shape[1:],
    output_pixel_size=sampleconfig.sample_structure.real_space_pixel_size,
    aperture_types=("OH",),
)
magnetic_pattern_oh = magnetic_pattern * oh_mask

# The canonical writer uses singular pipeline-style aperture keys.
pipeline_aperture_config = {
    "aperture_types": apertures_types,
    "aperture_radii": apertures_radius,
    "aperture_centers": apertures_centers,
    "aperture_sigmas": apertures_sigma,
    "aperture_angles": apertures_angle,
    "aperture_ellipticities": apertures_ellipticity,
    "aperture_roughnesses": apertures_roughness,
    "aperture_roughness_modes": apertures_roughness_modes,
    "aperture_seeds": apertures_seed,
    "aperture_top_radius_factors": apertures_top_radius_factor,
}

# This notebook generates the magnetic pattern on the full field, while the
# aperture and dielectric tensor use their ROI-aware representations.
canonical_metadata = HologramPipeline.build_precomputed_metadata(
    xray_config=xrayconfig,
    detector_config=detectorconfig,
    beamstop_config=beamstop_config,
    sample_config=sampleconfig,
    magnetic_pattern_config=magnetic_pattern_config,
    aperture_config=front_aperture_config,
    illumination_config=illuminationconfig,
    propagator_config=samplepropagationconfig,
    use_roi=True,
    magnetic_pattern_use_roi=False,
    dielectric_tensor_use_roi=True,
    dielectric_tensor_compact=True,
    save_detected_hologram_without_beamstop=save_detected_hologram_without_beamstop,
)

In [ ]:
# Use the shared pipeline writer so this file has exactly the same layout as a sweep.
export_pipeline = HologramPipeline(
    config=HologramPipelineConfig(recipe=recipe, oversampling=oversampling),
    ranges=HologramPipelineRanges(),
    output_path=output_path,
    n_samples=1,
    verbose=False,
)
written_path = export_pipeline.write_precomputed_result(
    hologram_config=hologram_config,
    detector_config=detectorconfig,
    metadata=canonical_metadata,
    aperture_config=pipeline_aperture_config,
    supportmask=supportmask,
    magnetic_pattern_oh=magnetic_pattern_oh,
    overwrite=True,
)
print(f"Wrote canonical pipeline HDF5 file: {written_path}")